# **LAB Deep Learning Session 5**
- ini cell 1-10 codingan sesi 5 kemarin

In [3]:
import tensorflow as tf

In [4]:
dataset = tf.data.TextLineDataset("dataset/ind.txt")

In [5]:
VOCAB_SIZE = 7000      
ENG_SEQ_LENGTH = 16
IND_SEQ_LENGTH = 16    
EMBEDDING_DIM = 256
BATCH_SIZE = 128       
HIDDEN_UNITS = 256     

In [6]:
eng_vec_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=ENG_SEQ_LENGTH
)

ind_vec_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=IND_SEQ_LENGTH
)

In [7]:
def preprocess(text):
    text = tf.strings.split(text, "\t")
    eng = text[0]
    ind = "starttoken " + text[1] + " endtoken"
    return eng, ind

def vectorize(text):
    text = tf.strings.split(text, "\t")
    eng = text[0]
    start = "starttoken " + text[1]     
    end = text[1] + " endtoken"         
    return {
        "input_1": eng_vec_layer(eng),      
        "input_2": ind_vec_layer(start),    
    }, ind_vec_layer(end)

In [8]:
preprocessed_dataset = dataset.map(preprocess)

eng_adapt_dataset = preprocessed_dataset.map(lambda eng, ind: eng).batch(1024)
eng_vec_layer.adapt(eng_adapt_dataset)

ind_adapt_dataset = preprocessed_dataset.map(lambda eng, ind: ind).batch(1024)
ind_vec_layer.adapt(ind_adapt_dataset)

In [9]:
n_data = int(dataset.reduce(0, lambda count, _: count + 1))   
n_val = int(n_data * 0.2)                                     

shuffled = dataset.shuffle(VOCAB_SIZE, seed=8)
val_raw, train_raw = shuffled.take(n_val), shuffled.skip(n_val)

train_dataset = (train_raw.map(vectorize).cache().shuffle(4096).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_raw.map(vectorize).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [10]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)  
        self.LSTM = tf.keras.layers.LSTM(hidden_units, return_sequences=True)

    def call(self, x):
        x = self.embedding(x)     
        return self.LSTM(x)